In [3]:
import pandas as pd
import math
from collections import defaultdict

def calculate_local_slab_forces(DL, LL, Lx, Lz, theta_deg):
    """Calculates the 6-DOF forces for a slab pitching along the X-axis (East/West)."""
    # Using the signed angle dictates the direction of the slope
    theta = math.radians(theta_deg)
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)
    
    # Gravity acts downward; we use the absolute value of the angle for vertical load distribution
    abs_cos_t = abs(cos_t)
    w_total = DL + LL * abs_cos_t
    w_perp = w_total * abs_cos_t
    w_para = w_total * math.sin(math.radians(abs(theta_deg)))
    
    # Yield Line Distribution (Purely geometric based on X and Z lengths)
    ratio = max(Lx, Lz) / min(Lx, Lz)
    if ratio > 2.0:
        if Lz > Lx: w_eq_Lx, w_eq_Lz = 0.0, w_total * Lx / 2
        else:       w_eq_Lz, w_eq_Lx = 0.0, w_total * Lz / 2
    else:
        if Lz >= Lx:
            w_eq_Lx = w_total * Lx / 4
            w_eq_Lz = (w_total * Lx / 2) * (1 - (Lx / (2 * Lz)))
        else:
            w_eq_Lz = w_total * Lz / 4
            w_eq_Lx = (w_total * Lz / 2) * (1 - (Lz / (2 * Lx)))
            
    # Rafter now spans Lx (X-axis). Eave now spans Lz (Z-axis).
    r_OP = w_eq_Lx * (w_perp / w_total) if w_total > 0 else 0
    e_OP = w_eq_Lz * (w_perp / w_total) if w_total > 0 else 0
    
    # In-plane sliding force carries the sign of the angle to push East or West
    e_IP_signed = w_eq_Lz * (w_total * sin_t / w_total) if w_total > 0 else 0
    
    # Moment Equations mapped to the new axes
    M_raf = r_OP * Lx**2 / 12       # Rafter bends along X, moment acts around Z-axis
    M_eave_OP = e_OP * Lz**2 / 12   # Eave bends along Z, moment acts around X-axis
    M_eave_IP = e_IP_signed * Lz**2 / 12 
    
    Fx, Fy, Fz = 0.0, - (w_total * Lx * Lz) / 4.0, 0.0
    
    # Your custom Global Grid Node Dictionary
    # sz = North/South multiplier | sx = West/East multiplier
    local_nodes = {
        'Node_1': (-1, -1),
        'Node_2': (-1, 1),
        'Node_3': (1, -1),
        'Node_4': (1, 1)
    }
    
    local_forces = {}
    for loc_node, (sz, sx) in local_nodes.items():
        # Global Moments correctly aligned to cancel out at concurrent nodes
        Mx = sz * (M_eave_OP * cos_t - M_eave_IP * sin_t)
        My = sz * (M_eave_OP * sin_t + M_eave_IP * cos_t)
        Mz = sx * M_raf
        
        local_forces[loc_node] = {'Fx': Fx, 'Fy': Fy, 'Fz': Fz, 'Mx': Mx, 'My': My, 'Mz': Mz}
        
    return local_forces

# --- ASSEMBLY PROCESS ---

try:
    df_in = pd.read_csv('input_slabs.csv')
except FileNotFoundError:
    print("Error: 'input_slabs.csv' not found. Please ensure the file exists in the directory.")
    exit()

global_nodes = defaultdict(lambda: {'Fx': 0.0, 'Fy': 0.0, 'Fz': 0.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0})

for index, row in df_in.iterrows():
    floor_id = row['Floor']
    local_forces = calculate_local_slab_forces(row['DL'], row['LL'], row['Lx'], row['Lz'], row['Theta_deg'])
    
    for i in range(1, 5):
        local_key = f'Node_{i}'
        node_id = row[f'Node_{i}_ID']
        unique_key = (floor_id, node_id)
        forces = local_forces[local_key]
        
        global_nodes[unique_key]['Fx'] += forces['Fx']
        global_nodes[unique_key]['Fy'] += forces['Fy']
        global_nodes[unique_key]['Fz'] += forces['Fz']
        global_nodes[unique_key]['Mx'] += forces['Mx']
        global_nodes[unique_key]['My'] += forces['My']
        global_nodes[unique_key]['Mz'] += forces['Mz']

df_global = pd.DataFrame.from_dict(global_nodes, orient='index')
df_global.index = pd.MultiIndex.from_tuples(df_global.index, names=['Floor', 'Archi_Node'])
df_global = df_global.sort_index()

print("--- SUMMED GLOBAL NODAL FORCES BY FLOOR ---")
print(df_global.to_string())

output_filename = 'loads_slab.csv'
df_global.to_csv(output_filename)
print(f"\nSuccess! Exported results to '{output_filename}'")

--- SUMMED GLOBAL NODAL FORCES BY FLOOR ---
                   Fx         Fy   Fz            Mx           My            Mz
Floor Archi_Node                                                              
2F    B'-3        0.0  -6.743880  0.0   2585.154000     0.000000  -1055.292333
      B'-4        0.0  -6.743880  0.0  -2585.154000     0.000000  -1055.292333
      B-2         0.0 -15.562800  0.0   5471.296875     0.000000  -6614.190000
      B-3         0.0 -15.562800  0.0  -5471.296875     0.000000  -6614.190000
      B1-4        0.0 -12.698813  0.0      0.000000     0.000000 -15873.515625
      B1-5        0.0 -12.698813  0.0      0.000000     0.000000 -15873.515625
      C-2         0.0 -35.664750  0.0  10942.593750     0.000000  -6619.593750
      C-3         0.0 -58.490190  0.0  -5556.135750     0.000000 -17357.445417
      C-4         0.0 -22.825440  0.0  -5386.458000     0.000000 -10737.851667
      D-2         0.0 -20.101950  0.0   5471.296875     0.000000  13233.783750
      D-

In [4]:
import csv

# 1. Your raw text block pasted directly into the script
mapping_text = """
node	Archi_Node
16	B'-3
17	B'-4
7	B-2
8	B-3
22	B1-4
23	B1-5
32	C-2
33	C-3
34	C-4
50	D-2
51	D-3
52	D-4
53	D-5
18	B'-4
19	B'-5
10	B-2
11	B-3
105	B-3'
36	C-2
37	C-3
104	C-3'
38	C-4
39	C-5
54	D-2
55	D-3
56	D-4
57	D-5
106	B'-3'
20	B'-4
21	B'-5
13	B-2
14	B-3
103	B-3'
73	B1-2
75	B1-3
40	C-2
41	C-3
102	C-3'
42	C-4
43	C-5
74	C1-2
76	C1-3
58	D-2
59	D-3
60	D-4
61	D-5
80	B-2
107	B-3'
82	B-4
88	B1-2
108	B1-3'
92	B1-4
89	C1-2
90	C1-2A
110	C1-3'
93	C1-4
83	D-2
109	D-2A
111	D-3'
85	D-4
94	uB1-2
97	uB1-2A
98	uB1-4
95	uC1-2
96	uC1-2A
99	uC1-4
"""

# 2. Parse the text block into a Python dictionary automatically
node_mapping = {}
lines = mapping_text.strip().split("\n")
for line in lines[1:]:  # Skip the header row ("node    Archi_Node")
    parts = line.split()
    if len(parts) >= 2:
        node_num = parts[0]
        archi_node = parts[1]
        node_mapping[archi_node] = node_num

# 3. Read the existing CSV file as text lines
csv_filename = "loads_slab.csv"

with open(csv_filename, mode="r", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

# 4. Find where 'Archi_Node' is located in your CSV header
if "Archi_Node" not in header:
    raise ValueError("Could not find 'Archi_Node' column in the CSV file.")
archi_idx = header.index("Archi_Node")

# 5. Insert the new 'node' column at the front of the file
new_header = ["node"] + header
new_rows = []

for row in rows:
    archi_val = row[archi_idx]
    # Fetch the node number from our text-parsed dictionary (defaults to empty string if unmapped)
    node_val = node_mapping.get(archi_val, "") 
    new_rows.append([node_val] + row)

# 6. Save back to the original file in place
with open(csv_filename, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(new_header)
    writer.writerows(new_rows)

print(f"Successfully parsed text mapping and updated {csv_filename} in place!")

Successfully parsed text mapping and updated loads_slab.csv in place!


In [6]:
import numpy as np
import pandas as pd
from collections import defaultdict

# 1. Define Material Density for Concrete
# 2.4e-8 kN/mm^3 
CONCRETE_DENSITY = 2.4e-8 

# 2. Load Properties CSV
csv_file_path = 'properties.csv'
try:
    df_props = pd.read_csv(csv_file_path)
except FileNotFoundError:
    print(f"Error: '{csv_file_path}' not found. Check filename.")
    exit()

# Dictionary to hold the summed forces for each node
# Format: {Node_ID: [Fx, Fy, Fz, Mx, My, Mz]}
nodal_loads = defaultdict(lambda: np.zeros(6))

print("Calculating Concrete Element Self-Weights and Equivalent Nodal Forces from CSV...")

# 3. Process Elements
for index, row in df_props.iterrows():
    i_node = int(row['Node_I'])
    j_node = int(row['Node_J'])
    
    A = float(row['Area (mm2)'])
    
    # Extract coordinates directly from the row
    xi, yi, zi = float(row['Xi']), float(row['Yi']), float(row['Zi'])
    xj, yj, zj = float(row['Xj']), float(row['Yj']), float(row['Zj'])
    
    dx, dy, dz = xj - xi, yj - yi, zj - zi
    
    # We recalculate L here just to ensure precision matches the dx, dy, dz vectors
    L = np.sqrt(dx**2 + dy**2 + dz**2) 
    
    if L == 0 or A == 0:
        continue
        
    # Total weight in kN: Area [mm^2] * L [mm] * Density [kN/mm^3]
    total_weight = A * L * CONCRETE_DENSITY
    
    # Check if the member is a perfectly vertical column
    if abs(dx) < 1e-6 and abs(dz) < 1e-6:
        # Entire load goes to the bottom node
        bottom_node = i_node if yi < yj else j_node
        nodal_loads[bottom_node][1] -= total_weight
        
    else:
        # --- HORIZONTAL / ANGLED BEAMS (Roof/Floor Members) ---
        # 1. Establish the Local Coordinate Transformation Matrix (R)
        rx = np.array([dx/L, dy/L, dz/L])
        v = np.array([0.0, 1.0, 0.0]) # Standard vertical reference vector
        
        z_vec = np.cross(rx, v)
        rz = z_vec / np.linalg.norm(z_vec)
        ry = np.cross(rz, rx)
        
        R = np.vstack([rx, ry, rz]) # 3x3 Rotation Matrix
        
        # 2. Transform the Global Gravity Vector into Local Beam Coordinates
        q_global = np.array([0.0, -total_weight / L, 0.0]) # Distributed load (kN/mm)
        q_local = R @ q_global
        
        # 3. Calculate Equivalent Nodal Loads (ENL) in Local Coordinates
        F_local_I = q_local * L / 2.0
        F_local_J = q_local * L / 2.0
        
        # ENL moments (Fixed End Moments converted to actions ON the node)
        # Moments will be in kN-mm
        M_local_I = np.array([0.0, -q_local[2] * L**2 / 12.0,  q_local[1] * L**2 / 12.0])
        M_local_J = np.array([0.0,  q_local[2] * L**2 / 12.0, -q_local[1] * L**2 / 12.0])
        
        # 4. Transform Local ENLs back to Global Coordinates
        F_global_I = R.T @ F_local_I
        M_global_I = R.T @ M_local_I
        F_global_J = R.T @ F_local_J
        M_global_J = R.T @ M_local_J
        
        # 5. Add to the concurrent nodal summation
        nodal_loads[i_node][:3] += F_global_I
        nodal_loads[i_node][3:] += M_global_I
        nodal_loads[j_node][:3] += F_global_J
        nodal_loads[j_node][3:] += M_global_J

# 4. Format Output to DataFrame
output_data = []
for node_id in sorted(nodal_loads.keys()):
    forces = nodal_loads[node_id]
    
    output_data.append({
        'node': node_id,
        'Fx': round(forces[0], 6),
        'Fy': round(forces[1], 6),
        'Fz': round(forces[2], 6),
        'Mx': round(forces[3], 6),
        'My': round(forces[4], 6),
        'Mz': round(forces[5], 6)
    })

df_loads = pd.DataFrame(output_data)

# Print Preview
print("\n--- SUMMED GLOBAL SELF-WEIGHT NODAL LOADS (CONCRETE) (kN, kN-mm) ---")
print(df_loads.head(10).to_string(index=False))

# Export to CSV
output_filename = 'loads_sw.csv'
df_loads.to_csv(output_filename, index=False)
print(f"\nSuccess! Full load table exported to '{output_filename}'")

Calculating Concrete Element Self-Weights and Equivalent Nodal Forces from CSV...

--- SUMMED GLOBAL SELF-WEIGHT NODAL LOADS (CONCRETE) (kN, kN-mm) ---
 node  Fx      Fy  Fz      Mx  My      Mz
    1 0.0 -12.528 0.0  3240.0 0.0 -3686.4
    2 0.0 -16.032 0.0 -1108.4 0.0 -3686.4
    3 0.0 -11.712 0.0 -2131.6 0.0 -3686.4
    4 0.0  -6.960 0.0     0.0 0.0     0.0
    5 0.0  -6.960 0.0     0.0 0.0     0.0
    6 0.0  -6.960 0.0     0.0 0.0     0.0
    7 0.0 -15.408 0.0  3240.0 0.0 -3686.4
    8 0.0 -13.176 0.0 -1108.4 0.0  -774.4
    9 0.0  -8.856 0.0 -2131.6 0.0  -774.4
   10 0.0 -15.408 0.0  3240.0 0.0 -3686.4

Success! Full load table exported to 'loads_sw.csv'
